# Single Pixel Camera + Gaussian Splatting (versión simple, 2D)

Este notebook es una versión **autocontenida** del proyecto: no usa GitHub, ni Drive, ni el dataset `truck`, ni GPU obligatoria. Todo corre en Google Colab (o en su computador) con `numpy`, `scipy`, `scikit-image` y `torch`.

**Idea.** Se simula una cámara de un solo píxel con patrones Hadamard, igual que en el notebook original. Luego se comparan dos formas de reconstruir la imagen con la misma cantidad de mediciones:

1. **Hadamard clásico:** se conservan los coeficientes de baja frecuencia medidos y el resto se deja en cero.
2. **Gaussian Splatting 2D:** la imagen se representa como una suma de $K$ gaussianas (posición, escala, rotación e intensidad) y sus parámetros se optimizan para que, al aplicarles el **mismo modelo de medición** $y_k = \langle \phi_k, x \rangle$, reproduzcan las mediciones.

**Pregunta (versión 2D de la del proyecto):** ¿a partir de qué ratio de mediciones $r$ la reconstrucción deja de degradarse de forma notoria, y ayuda la representación con gaussianas?

**Alcance:** una imagen, una base Hadamard, una semilla, sin ruido físico. Es una demostración pedagógica, no un resultado general.

## 1. Configuración

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.linalg import hadamard
from skimage import data, transform
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SIZE = 64                            # potencia de dos (requisito de Hadamard)
RATIOS = [0.05, 0.10, 0.25, 0.50]    # fracción de mediciones respecto a SIZE*SIZE
N_GAUSSIANS = 150                    # número de splats
ITERATIONS = 600                     # pasos de optimización por ratio
IMAGE_PATH = None                    # p. ej. "/content/mi_foto.jpg"; None = imagen de ejemplo
SEED = 449

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Dispositivo:", DEVICE)

## 2. Imagen de entrada

Se usa una imagen en gris de ejemplo incluida en `scikit-image`. Para usar una foto propia (por ejemplo un fotograma de `truck`), suba el archivo a Colab y escriba su ruta en `IMAGE_PATH`.

In [ ]:
if IMAGE_PATH is None:
    raw = data.camera().astype(np.float32) / 255.0
else:
    from PIL import Image
    raw = np.asarray(Image.open(IMAGE_PATH).convert("L"), dtype=np.float32) / 255.0

img = transform.resize(raw, (SIZE, SIZE), anti_aliasing=True).astype(np.float32)
plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.title(f"Imagen original {SIZE}x{SIZE}")
plt.axis("off")
plt.show()

## 3. Cámara de un solo píxel con patrones Hadamard

Es la misma matemática del notebook original: la base Hadamard 2D se ordena por frecuencia espacial y se conservan los primeros $n = r \cdot N$ coeficientes. Cada coeficiente equivale a una medición de un solo píxel.

In [ ]:
h = (hadamard(SIZE) / np.sqrt(SIZE)).astype(np.float32)
changes = np.sum(h[:, 1:] != h[:, :-1], axis=1)
order_1d = np.argsort(changes, kind="stable")
pairs = [(i, j) for i in order_1d for j in order_1d]
pairs.sort(key=lambda ij: (changes[ij[0]] + changes[ij[1]], max(changes[ij[0]], changes[ij[1]])))
Ht = torch.tensor(h, device=DEVICE)


def measure(x, n):
    """y_k = <phi_k, x> para las n frecuencias más bajas."""
    coeffs = Ht @ x @ Ht.T
    idx = torch.tensor(pairs[:n], device=DEVICE)
    return coeffs[idx[:, 0], idx[:, 1]], idx


def reconstruct_hadamard(y, idx):
    """Reconstrucción clásica: coeficientes no medidos = 0."""
    coeffs = torch.zeros(SIZE, SIZE, device=DEVICE)
    coeffs[idx[:, 0], idx[:, 1]] = y
    return (Ht.T @ coeffs @ Ht).clamp(0, 1)


x_true = torch.tensor(img, device=DEVICE)
print("Mediciones completas:", SIZE * SIZE)

## 4. Gaussian Splatting 2D diferenciable

Cada gaussiana tiene centro, escala (en $x$ e $y$), ángulo de rotación e intensidad. La imagen es la suma de todas. Se optimiza con Adam para minimizar el error entre las mediciones Hadamard **simuladas desde los splats** y las mediciones reales del sensor. Esto es lo que conecta la cámara de un solo píxel con el splatting: el modelo directo forma parte de la pérdida.

In [ ]:
gy, gx = torch.meshgrid(
    torch.arange(SIZE, device=DEVICE).float(),
    torch.arange(SIZE, device=DEVICE).float(),
    indexing="ij",
)
grid = torch.stack([gx, gy], dim=-1)  # (H, W, 2)


def fit_gaussians(y, idx, k=N_GAUSSIANS, iters=ITERATIONS):
    mu = (torch.rand(k, 2, device=DEVICE) * SIZE).requires_grad_()
    log_s = torch.full((k, 2), float(np.log(2.5)), device=DEVICE).requires_grad_()
    theta = torch.zeros(k, device=DEVICE).requires_grad_()
    amp = torch.full((k,), 0.05, device=DEVICE).requires_grad_()

    def render():
        c, s = torch.cos(theta), torch.sin(theta)
        rot = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)  # (k,2,2)
        d = grid[None] - mu[:, None, None, :]
        d = torch.einsum("kij,khwj->khwi", rot.transpose(1, 2), d)
        q = (d / torch.exp(log_s)[:, None, None, :]) ** 2
        return (amp[:, None, None] * torch.exp(-0.5 * q.sum(-1))).sum(0)

    opt = torch.optim.Adam([
        {"params": [mu], "lr": 0.2},
        {"params": [log_s], "lr": 0.03},
        {"params": [theta], "lr": 0.03},
        {"params": [amp], "lr": 0.02},
    ])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    for _ in range(iters):
        opt.zero_grad()
        coeffs = Ht @ render() @ Ht.T               # mismo modelo directo del sensor
        loss = ((coeffs[idx[:, 0], idx[:, 1]] - y) ** 2).mean() / (y ** 2).mean()
        loss.backward()
        opt.step()
        sched.step()
    return render().detach().clamp(0, 1), loss.item()

## 5. Barrido de ratios

Se varía únicamente el ratio $r$. Imagen, base, semilla y número de gaussianas quedan fijos. En CPU tarda unos minutos; con GPU es bastante más rápido.

In [ ]:
def metrics(a, b):
    a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    return (
        peak_signal_noise_ratio(a, b, data_range=1.0),
        structural_similarity(a, b, data_range=1.0),
    )


rows, recon = [], {}
for r in RATIOS:
    n = int(round(r * SIZE * SIZE))
    y, idx = measure(x_true, n)
    had = reconstruct_hadamard(y, idx)
    t0 = time.time()
    gs, final_loss = fit_gaussians(y, idx)
    elapsed = time.time() - t0
    recon[r] = (had, gs)
    for name, rec in [("Hadamard clásico", had), ("Gaussian Splatting 2D", gs)]:
        psnr, ssim = metrics(x_true, rec)
        rows.append({"ratio": r, "mediciones": n, "método": name, "PSNR (dB)": round(psnr, 2), "SSIM": round(ssim, 3)})
    print(f"r={r:.2f} ({n} mediciones) listo en {elapsed:.0f} s | pérdida de medición: {final_loss:.5f}")

results = pd.DataFrame(rows)
results.pivot(index=["ratio", "mediciones"], columns="método", values=["PSNR (dB)", "SSIM"])

## 6. Resultados visuales

In [ ]:
fig, axes = plt.subplots(3, len(RATIOS) + 1, figsize=(3 * (len(RATIOS) + 1), 9))
axes[0, 0].imshow(img, cmap="gray", vmin=0, vmax=1); axes[0, 0].set_title("Original")
axes[1, 0].axis("off"); axes[2, 0].axis("off")
for col, r in enumerate(RATIOS, start=1):
    had, gs = recon[r]
    axes[0, col].imshow(had.cpu(), cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"Hadamard, r={r:.2f}")
    axes[1, col].imshow(gs.cpu(), cmap="gray", vmin=0, vmax=1)
    axes[1, col].set_title(f"Splatting, r={r:.2f}")
    axes[2, col].imshow((gs - x_true).abs().cpu(), cmap="magma", vmin=0, vmax=0.5)
    axes[2, col].set_title("|error| splatting")
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for name, g in results.groupby("método"):
    ax[0].plot(g["ratio"], g["PSNR (dB)"], marker="o", label=name)
    ax[1].plot(g["ratio"], g["SSIM"], marker="o", label=name)
ax[0].set_xlabel("ratio de mediciones r"); ax[0].set_ylabel("PSNR (dB)"); ax[0].legend()
ax[1].set_xlabel("ratio de mediciones r"); ax[1].set_ylabel("SSIM")
plt.tight_layout()
plt.show()

## 7. Cómo leer los resultados

- **PSNR (dB):** más alto es mejor. **SSIM (0 a 1):** más cerca de 1 es mejor.
- Con pocas mediciones, el problema está mal condicionado: hay muchas imágenes compatibles con los mismos datos. Las gaussianas actúan como un a priori de imagen suave y compacta, lo que puede ayudar donde la reconstrucción Hadamard solo produce una versión borrosa.
- Con muchas mediciones, la reconstrucción Hadamard ya conserva casi toda la información y el número fijo de gaussianas pasa a ser el límite de calidad. Es esperable que ahí el splatting deje de ganar.
- Este notebook ejecuta un experimento con una sola imagen y una sola semilla. Para sacar conclusiones conviene repetir con varias imágenes y semillas, y variar `N_GAUSSIANS`.

## 8. Extensiones posibles

1. Usar un fotograma de `truck` (o de otra escena) en `IMAGE_PATH`.
2. Extender a RGB (un splat con tres intensidades, o tres canales por separado).
3. Agregar ruido gaussiano al detector y ver cómo cambia el ratio mínimo.
4. Comparar contra una línea base con regularización TV o LASSO.
5. Pasar a 3D: reconstruir varias vistas con este método y usarlas como entrada de 3DGS, que es lo que hace el notebook original. Esa etapa sí requiere GPU CUDA y compilar los submódulos del repositorio.